# MedGemma recall benchmark on MDACE — Colab Runner (T4 free tier)

How much of the billed evidence does **google/medgemma-4b-it** recover from a
clinical note, zero-shot? One question, one input file, one number quoted with
the volume it was bought with.

**Recall is the metric.** Precision appears as an explicit false-positive count
rather than as a ratio, because MDACE annotates evidence only for codes that were
actually billed and a correct extraction of an unbilled condition is not a model
error.

Gold is the **union of three columns** per billed code: the phrase the coder
highlighted, the ICD code description, and the SNOMED terms the file ships.
A prediction matching any of them recalls that row. Matching runs a four-level
ladder from exact string equality up to biomedical embeddings, and each level is
a superset of the one above so the gain from each is attributable.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

---

## ⚠️ THIS NOTEBOOK HANDLES REAL PATIENT DATA

MDACE is built on **MIMIC-III** notes — credentialed PhysioNet data.

- The file you upload **contains note text**. Never commit it, never paste note
  content into a chat, an issue or a message.
- Everything the run writes to Drive except `per_note.jsonl` quotes note text:
  the extracted findings, and every pair the ladder newly accepted at each level.
- `results/mdace_recall_*.md` and `.json` are aggregate-only — counts, rates and
  thresholds — and are safe to share.


## 1. Confirm the T4 GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Install dependencies

`sentence-transformers` is here for **L4** of the matching ladder, the only level
that reaches abbreviations — `CHF` scores 0.22 against `congestive heart failure`
on characters, so no string rule will ever find it. Without the package the
ladder simply stops at L3 and the report says so; it does not fail.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes tqdm huggingface_hub sentence-transformers

## 3. Hugging Face login (gated model)

Accept the license at https://huggingface.co/google/medgemma-4b-it, then paste a
token from https://huggingface.co/settings/tokens. Read via `getpass`, so it is
never stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token (input hidden): '))

## 4. Get the project code — and re-run this cell to pull updates

Code only — the repo is public and contains **no** patient data.

**This cell is idempotent: run it again any time to pull the latest code.** It
clones on first use and hard-resets to the remote branch afterwards, so there is
no separate "pull" step to remember and no way to end up running a half-updated
checkout. It prints the commit you ended up on — check that before trusting a
run, for the same reason the report prints the prompt hash.

The reset discards local edits inside the repo but **never touches the uploaded
input file**: that lives in gitignored `data/samples/`, so pulling does not cost
you a re-upload the way a fresh clone would.

In [ ]:
import os

REPO = '/content/medgemma-ner-eval'
BRANCH = 'mdace-recall-benchmark'
URL = 'https://github.com/shifat514/medgemma-ner-eval.git'

if os.path.isdir(os.path.join(REPO, '.git')):
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} reset -q --hard origin/{BRANCH}
    print('pulled')
else:
    !git clone -q {URL} {REPO}
    !git -C {REPO} checkout -q {BRANCH}
    print('cloned')

%cd {REPO}
!git log -1 --oneline
!git status --short --branch | head -1

## 5. Mount Drive for run output

**This is what makes the run survivable.** Colab's own disk is wiped when the
runtime recycles; a Drive folder is not. `RECALL_OUTPUT_DIR` redirects the
per-note state there, so a disconnect costs only the note in flight.

| file | contents | shareable |
|---|---|---|
| `per_note.jsonl` | integer counts only | yes |
| `findings.jsonl` | the `{span, name}` lists the model produced | **no — note text** |
| `new_pairs_L*.jsonl` | what each ladder level newly accepted | **no — note text** |
| `raw_replies.jsonl` | raw model output, smoke run only | **no — note text** |

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/mdace-recall-outputs'
os.makedirs(OUT, exist_ok=True)
os.environ['RECALL_OUTPUT_DIR'] = OUT
print('run output ->', OUT)
print('existing run dirs:', os.listdir(OUT) or '(none yet)')

## 6. Upload the input file BY HAND

There is **no sample-building step** for this benchmark. It reads one file,
`8-07-mdace-ner-eval_sample_100-LOCAL.jsonl`, which embeds `note_text` on every
row — so there is no join and no separate notes file.

Get it from `s3://zeda-mimic-dataset/eval_datasets/` on the machine that holds
your credentials, run the cell below and pick it.

The file goes to this VM's ephemeral disk, not to Drive — note text should not
outlive the runtime.

In [ ]:
import json, os, shutil
from google.colab import files

os.makedirs('data/samples', exist_ok=True)
DEST = 'data/samples/mdace_recall_input.jsonl'
os.environ['RECALL_SAMPLE_FILE'] = DEST

if os.path.exists(DEST):
    print('already present:', DEST)
else:
    uploaded = files.upload()  # choose 8-07-mdace-ner-eval_sample_100-LOCAL.jsonl
    name = next(iter(uploaded))
    if name != DEST:
        shutil.move(name, DEST)

# Sanity check WITHOUT printing note text.
from src.datasets.mdace_recall import build_notes
records, stats = build_notes(DEST)
print(f'rows            {stats["n_rows"]:>5}   (expect 100)')
print(f'notes           {stats["n_notes"]:>5}   (expect 24)')
print(f'distinct codes  {stats["n_codes"]:>5}   (expect 91)')
print(f'chunks to run   {stats["n_chunks"]:>5}   (expect 82)')
print(f'accepted forms  {stats["forms_combined"]:>5}   '
      f'(evidence {stats["forms_by_source"]["evidence"]}, '
      f'description {stats["forms_by_source"]["description"]}, '
      f'snomed {stats["forms_by_source"]["snomed"]})')
print(f'median accepted forms per row: {stats["accept_median"]:.0f} '
      f'(min {stats["accept_min"]}, max {stats["accept_max"]})')

## 7. Harness check — no GPU, ~10 seconds

Feeds the gold accept-sets back through chunking, parsing, normalization and
matching with **no model involved**.

**Every source must read 1.0000 at L1, and the combined matching must show zero
false positives.** The cell prints PASS or FAIL. Anything less is a bug in the
harness, not a result, and there is no point burning GPU hours until it is fixed.
This caught real bugs twice on the previous branch and costs ten seconds.

Per-source false positives will be non-zero and that is correct rather than a
defect: per-source FP means "matched nothing *in that source*", so a finding that
names the catalogue wording is a false positive on the evidence-text line by
construction.

In [ ]:
!python -m src.evaluate_recall --oracle

## 8. Smoke test — the 3 longest notes, 30–50 min

These are **not** the first 3 in the file. Records are ordered longest-first, so
this runs the multi-chunk path — where OOM and truncation actually live — before
committing to the full run.

**Budget it properly: this is 24 of the 82 chunks, 29% of the whole benchmark.**
It is a big smoke test because the benchmark is small. On a T4 expect 30–50 min
of generation, plus 5–10 min the first time for the model download.

**None of it is thrown away.** The run directory is keyed on model, chunk
geometry, token cap and prompt hash — not on how many notes you asked for — so
the full run in cell 10 resumes these three notes and only pays for the
remaining 58 chunks.

**The progress bar counts notes, not chunks.** With three notes it reads `0/3`
for the first ten minutes or so and `1/3` for the next fifteen. That is normal
and is not a hang; the elapsed timer climbing is the signal that it is alive.

**Watch for:**
- CUDA OOM → lower `--chunk-words` (try 300, then 250).
- `generation hit max_new_tokens` **> 0** → replies were truncated mid-JSON and
  recall is understated. The two-field output costs ~2x tokens per finding, so
  this matters more here than it did on the previous branch. The lever is output
  volume, not a higher cap.
- `standard name but no span` **high** → the model is not copying phrases
  verbatim, and the not-in-note check loses its denominator.
- `returned no usable JSON` **> 0** → check one raw reply in the next cell.

In [ ]:
!python -m src.evaluate_recall --smoke 3 --dump-replies

## 8b. The prompt A/B — ~50 min, and it settles a recurring argument

The `scoped` prompt names the categories to exclude, and it has grown a rule
every time a run found a new leak: medications, then vital signs, then lab
values, then blood products, then bare anatomy. There is no principle in that
list saying when it is finished, and the set of things a clinical note contains
that a coder does not bill is effectively unbounded.

`billable` tests the alternative: delete all four exclusions and replace them
with the criterion that actually *defines* gold — **would a medical coder assign
a code to this?** MDACE evidence is the phrase a coder highlighted to justify a
submitted code, so that is the target rather than a proxy for it.

The risk runs the wrong way, which is exactly why this is measured: recall is the
metric, and a model with a weak grasp of billability under-extracts. Right now
`scoped` over-extracts by roughly 17x, so there is room to lose volume before
scarcity binds.

**Read the volume rows before the recall row.** A prompt that extracts more
scores higher recall almost regardless of quality. What would count as a win for
`billable` is *recall held at materially lower volume*.

The two prompts hash differently, so they land in separate run directories and
neither can replay the other's numbers.

In [ ]:
!python -m src.evaluate_recall --smoke 3 --dump-replies --prompt scoped
!python -m src.evaluate_recall --smoke 3 --dump-replies --prompt billable
!python -m src.recall_compare

## 8c. The chunk-size experiment — ~45 min

The prompt A/B settled which prompt to use and **did not fix the real problem**.
Both variants extract 15-17x the gold: roughly 100 findings per note against 6.5
billed rows. That volume is what makes a recall of 1.0000 uninformative, and it
is why the token cap keeps being hit.

The prompt was never going to fix it. What triggers both the volume and the
repetition loop is **list length** — greedy decoding degenerates on long
structured output — so the lever is giving the model less to describe per call.

| chunk words | chunks (2 notes) | chunks (all 24) |
|---|---|---|
| 400 / 80 overlap | 17 | 82 |
| 250 / 50 overlap | 27 | 126 |

More calls, each writing less, so the wall clock is roughly a wash.

**What would count as a win at 250:** fewer findings per note, a lower
`repeated within their own reply` count, and fewer chunks cut at the cap — while
row recall holds. If recall drops sharply, 400 was right and the volume is real
rather than degenerate.

Both arms write separate results files now: the label carries the chunk geometry
as well as the prompt, so neither can overwrite the other.

In [ ]:
!python -m src.evaluate_recall --smoke 2 --dump-replies --chunk-words 400 --overlap-words 80
!python -m src.evaluate_recall --smoke 2 --dump-replies --chunk-words 250 --overlap-words 50
!python -m src.recall_compare

## 9. What did the model actually say?

Reply **shapes and counts** only. The replies quote note text, so raw text is not
printed by default — set `SHOW_ONE = True` to debug a parse failure, and clear
the cell output afterwards.

In [ ]:
import glob, json, os
from collections import Counter

SHOW_ONE = False  # True prints one raw reply — CONTAINS NOTE TEXT

paths = glob.glob(os.path.join(os.environ['RECALL_OUTPUT_DIR'], '*', 'raw_replies.jsonl'))
if not paths:
    print('no raw_replies.jsonl — run the smoke cell with --dump-replies')
else:
    rows = [json.loads(l) for l in open(sorted(paths)[-1], encoding='utf-8')]
    print('chunks:', len(rows))
    print('shapes:', dict(Counter(r['shape'] for r in rows)))
    kept = [r['n_kept'] for r in rows]
    print('findings kept per chunk: min %d, median %d, max %d'
          % (min(kept), sorted(kept)[len(kept) // 2], max(kept)))
    print('chunks yielding zero findings:', sum(1 for k in kept if k == 0))
    if SHOW_ONE:
        print('\n--- one raw reply (CONTAINS NOTE TEXT) ---\n')
        print(rows[0]['reply'][:2000])

## 10. The benchmark — 24 notes / 82 chunks

The smoke run's 3 notes are cached and skipped, so this pays for the remaining
**58 chunks** — roughly 65–95 min on a T4 at the rate cell 8 just showed you.
Divide cell 8's wall-clock by 24 to get your actual seconds-per-chunk, then
multiply by 58.
The smoke run's 3 notes are cached and skipped.

**If Colab disconnects, re-run this cell.** Finished notes are read back from
Drive and skipped.

**If it finishes in seconds, do not believe it.** Check the `run dir:` line for
the prompt hash — a fully cached run replays old numbers with no error. That is
the one trap this harness cannot remove, only make visible.

In [ ]:
!python -m src.evaluate_recall

## 11. Re-score without re-running the model

Scoring thresholds are not part of the run directory name, so changing one
re-uses the cached inference. This is how to see what a threshold is worth
instead of arguing about it.

Thresholds are printed in the report every time, never chosen silently.

In [ ]:
!python -m src.evaluate_recall --score-only --dice-min 0.7 --ratio-min 0.85
# and back to the defaults, which is what the committed report should hold
!python -m src.evaluate_recall --score-only

## 12. L5 — adjudicate what the ladder bought

L1 is exact string equality and needs no judge. Every level above it admits pairs
nobody has checked: L2 knowingly lets through `diabetes` inside *diabetes
insipidus* (a different disease) and `sepsis` inside *no evidence of sepsis* (a
negation).

This runs only on the pairs each level **newly** accepted, so the cost stays
proportional to what the ladder actually gained.

**`--judge medgemma` has the model under test grade its own matches.** It is free
on a box that already has the model loaded, and the summary says so out loud.
Use `--judge none` to write the questions out for a human or a stronger model
instead, then feed the answers back with `--verdicts`.

In [ ]:
!python -m src.recall_judge --judge medgemma

## 13. Read the report

Aggregate metrics only — no note text, no phrases. Safe to share.

**Read recall as the result, and never quote it bare.** With loose matching and
no volume control, a model that lists every phrase in the note scores near 1.00,
so the findings-per-note and not-in-note lines have to travel with it.

In [ ]:
import glob
from IPython.display import Markdown, display

for path in sorted(glob.glob('results/mdace_recall_*.md')):
    print('=' * 70, '\n', path, '\n', '=' * 70)
    display(Markdown(open(path, encoding='utf-8').read()))

## 14. Save the outputs

Copies the committable artifacts to Drive alongside the run state. The findings
and the per-level pair dumps are already on Drive from the run itself — those are
the files an audit or a later L5 pass consumes, so neither has to cost these GPU
hours twice.

In [ ]:
import glob, os, shutil

dest = os.path.join(os.environ['RECALL_OUTPUT_DIR'], 'results')
os.makedirs(dest, exist_ok=True)
for path in glob.glob('results/mdace_recall_*'):
    shutil.copy(path, dest)
    print('saved', os.path.basename(path))

print('\nrun state on Drive:')
for root, _dirs, filenames in os.walk(os.environ['RECALL_OUTPUT_DIR']):
    for name in filenames:
        full = os.path.join(root, name)
        print('  %8.1f KB  %s' % (os.path.getsize(full) / 1024,
                                  os.path.relpath(full, os.environ['RECALL_OUTPUT_DIR'])))